In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

In [ ]:
# RSI
data=yf.download('AAPL',start='2020-01-01',end='2024-01-01')
dt1=pd.DataFrame(data)
# dt1
dt1["diff"]=dt1['Close'].diff()
length=14
dt1["gain"]=dt1["diff"].apply(lambda x:x if x>0 else 0)
dt1["loss"]=dt1["diff"].apply(lambda x:-x if x<0 else 0)
dt1['ema_gain'] = dt1['gain'].ewm(span=length, min_periods=length).mean()
dt1['ema_loss'] = dt1['loss'].ewm(span=length, min_periods=length).mean()
dt1['rs']=dt1['ema_gain']/dt1['ema_loss']
dt1['rsi']=100-(100/(1+dt1["rs"]))
dt1
signals = []
postn = 0  # 0 = No hold

for i in range(len(dt1)):
    rsi = float(dt1["rsi"].iloc[i])  
    date = dt1.index[i]  
    signal = 0  
    if rsi>=70 and postn==0:
        signal=1
        postn=1
    elif rsi<=50 and postn==1:
        signal=-1
        postn=0
    elif rsi<=30 and postn==0:
        signal=2
        postn=-1
    elif rsi>=50 and postn==-1:
        signal=-2
        postn=0
    signals.append([date, signal])
signals_df=pd.DataFrame(signals,columns=["Date","Signal"])
signals_df.to_csv("rsisignal.csv",index=False)

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


In [4]:
#MACD
data1 = yf.download("AAPL", start="2020-01-01", end="2024-01-01")
df2 = pd.DataFrame(data1)
p1, p2, sl = 12, 26, 9
df2["12 DAY EMA"] = df2["Close"].ewm(span=p1, adjust=False).mean()
df2["26 DAY EMA"] = df2["Close"].ewm(span=p2, adjust=False).mean()
df2["MACD"] = df2["12 DAY EMA"] - df2["26 DAY EMA"]
df2["Signal Line"] = df2["MACD"].ewm(span=sl, adjust=False).mean()
signals = []
currentpostn = 0  
firstsign=False
for i in range(len(df2)):
    date = df2.index[i]
    macdval = df2["MACD"].iloc[i]
    signval = df2["Signal Line"].iloc[i]

    signal = 0  

    if macdval > signval and currentpostn != 1:
        signal = 1  
        currentpostn = 1
    elif macdval < signval and currentpostn != -1:
        signal = -1  
        currentpostn = -1
    if not firstsign and signal == -1:
        signal = 1
        currentpostn = 1
        firstsign = True  
    elif signal != 0:
        firstsign = True  


    signals.append([date, signal])
signal_df = pd.DataFrame(signals, columns=["Date", "Signal"])
signal_df.set_index("Date", inplace=True)
df2["Signal"] = signal_df["Signal"]
df2[["Close", "Signal"]].to_csv("MACD_Signals.csv", index=True)  


[*********************100%***********************]  1 of 1 completed


In [ ]:
#BB
data2=yf.download("BTC-USD",start="2020-01-01",end="2025-01-01")
df3=pd.DataFrame(data2)
df3["sma"]=df3["Close"].rolling(window=20,min_periods=20).mean()
df3["std"]=df3["Close"].rolling(window=20,min_periods=20).std()
df3["upperbound"]=df3["sma"]+(2*df3["std"])
df3["lowerbound"]=df3["sma"]-(2*df3["std"])
btcsign=[]
prevsignal=None
openpostn=bool(False)
df3.dropna(inplace=True)
df3.head(20)
for i in range(len(df3)):
    date=df3.index[i]
    smaval=float(df3["sma"].iloc[i])
    closeval=float(df3["Close"].iloc[i])
    upper=float(df3["upperbound"].iloc[i])
    lower=float(df3["lowerbound"].iloc[i])
    signal=0
    
    if closeval >= upper and not openpostn:
        signal = 1
        openpostn = True
        prevsignal = signal 

    elif closeval <= smaval and openpostn and prevsignal == 1:
        signal = -1
        openpostn = False  

    elif closeval <= lower and not openpostn:
        signal = 2
        openpostn = True
        prevsignal = signal  

    elif closeval > smaval and openpostn and prevsignal == 2:
        signal = -2
        openpostn = False
    btcsign.append([date,signal])
signals_df = pd.DataFrame(btcsign, columns=["date", "Signal"])
signals_df.set_index("date",inplace=True)
signals_df.to_csv("bb.csv",index=True)


[*********************100%***********************]  1 of 1 completed
C:\Users\tusha\AppData\Local\Temp\ipykernel_22060\4283734599.py:15: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  closeval=float(df3["Close"].iloc[i])


In [16]:
data = yf.download("BTC-USD", start="2020-01-01", end="2025-01-01")
df4 = pd.DataFrame(data)
df4["sma50"] = df4["Close"].rolling(window=50).mean()  
df4["sma200"] = df4["Close"].rolling(window=200).mean() 
df4["Signal"] = 0 
postn=0
signals = []
df4.dropna(inplace=True)
for i in range(len(df4)):
    date = df4.index[i]
    sma50 = df4["sma50"].iloc[i]
    sma200 = df4["sma200"].iloc[i]
    prevsma50 = df4["sma50"].iloc[i - 1]
    prevsma200 = df4["sma200"].iloc[i - 1]

    signal = 0
    if postn==0 and prevsma50 < prevsma200 and sma50 > sma200:
        signal = 1
        postn=1

    elif postn==0 and prevsma50 > prevsma200 and sma50 < sma200:
        signal = 2
        postn=-1

    elif postn==1 and sma50<sma200:
        signal=-1
        postn=0

    elif postn==-1 and sma50>sma200:
        signal=-2
        postn=0
    signals.append([date, signal])

signals_df4 = pd.DataFrame(signals, columns=["Date", "Signal"])
signals_df4.set_index("Date", inplace=True)
signals_df4.to_csv("sma.csv", index=True)


[*********************100%***********************]  1 of 1 completed
